# 16. Latent Thresholds — F5: statistical layer over the fused latent space (FOC-179)

Phase F5 of FOC-174: the **dictionary model's threshold logic ported to the latent space**. nb4
calibrates thresholds over per-variable fraud probabilities on the training rows and applies them
frozen to test; here the same calibration story is applied to anomaly scores computed over the fused
1280-d embedding (L2-normalized face(512)+text(384)+demo(384), stateless concat — nb15/`arms_fusion`).

**Pre-registered expectation (from F3/F4):** on the PRIMARY axis (cohort-random, customer-grouped;
13 test positives, chance 0.0133) *every* arm so far covers chance — the latent space itself carries
~no detectable signal at 5.3k transactions. A threshold layer cannot exceed its input
representation, so the null is expected to persist here. The deliverable is the machinery: the
threshold/scoring methods, evaluated through the identical `funs.py`/`fraud_pipeline` protocol as
the 13 existing arms on all three axes, cheaply comparable and ready for more data.

**Arms (all registered in the runner, scorers in `arms_latent.py`):**

| arm | score (higher = more anomalous) | reference fitted on |
|---|---|---|
| `latent-nn-dist` | distance to the 5th-nearest legitimate fitting row (self-matches excluded) | legit fit carve |
| `latent-centroid-dist` | Euclidean distance to the legitimate centroid | legit fit carve |
| `latent-cosine-centroid` | angular score `1 - cos` to the legitimate mean direction | legit fit carve |
| `latent-cluster-anom` | distance to the nearest k-means center (k=8) | legit fit carve, TRAIN ONLY |
| `latent-gmm-density` | negative log likelihood under PCA(64)+GMM (diagonal) | legit fit carve, TRAIN ONLY |
| `latent-logistic` | supervised logistic regression probability (Mateusz's F5 decision) | full fit carve |

**Leakage discipline (FOC-179):** every fitted object (neighbor index, centroid, k-means, PCA, GMM,
logistic weights) is fitted inside `fit()` on the rows the runner hands the estimator — the
stratified fit carve of the training side — never on the full pre-split frame. No clustering or
density model ever sees test rows. The runner freezes its own best-F1 threshold on a validation
carve, so every arm below is measured through the identical protocol as the 13 F0-F4 arms; the
scorers' own train-percentile operating points are reported separately in the calibration study.

In [ ]:
import platform
import sys

import numpy as np
import pandas as pd
import sklearn

print('python', sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__, '| sklearn', sklearn.__version__)
print('platform', platform.platform())

from pathlib import Path

for name in [
    '../data/all_trxns.csv',
    '../data/dim_customer.csv',
    '../data/face_embeddings.npz',
    '../data/demo_embeddings.npz',
]:
    path = Path(name)
    print('%-34s %s' % (name, 'OK' if path.exists() else 'MISSING'))

In [ ]:
import arms_fusion
import arms_latent
from fraud_pipeline import (
    ARMS,
    DEFAULT_RESULTS_PATH,
    _features_latent_pure,
    axis_split,
    load_enriched,
    load_results,
    print_comparison_table,
    run_arm_on_axis,
)

# Canonical enriched frame + labels from the unified runner (the nb7/nb8 data
# section, shared by every arm — loaded once, used by everything below).
enriched, y = load_enriched()
print(
    'fraud txns: %d of %d (%.2f%%) across %d unique customers'
    % (int(y.sum()), len(y), 100 * y.mean(), enriched['customer'].nunique())
)
missing = arms_fusion.check_dependencies()
print('dependency probe:', missing if missing else 'latent caches OK (no encoder loads)')

F5_ARMS = [
    'latent-nn-dist',
    'latent-centroid-dist',
    'latent-cosine-centroid',
    'latent-cluster-anom',
    'latent-gmm-density',
    'latent-logistic',
]
assert all(arm in ARMS for arm in F5_ARMS), 'F5 arms must be registered in the runner'
frame_check = _features_latent_pure(enriched)
assert list(frame_check.columns) == list(arms_fusion.FUSED_FEATURES)
assert frame_check.shape == (len(enriched), 1280)
print('F5 arms registered: %d | pure latent frame %s' % (len(F5_ARMS), frame_check.shape))

## Methods — the dictionary model's threshold story, ported

nb4's dictionary model builds per-variable lookup tables on the training rows, scores each
transaction by aggregating per-variable fraud probabilities, and **calibrates its thresholds on the
training rows only** (a 3-threshold F1 grid; `DictionaryRateEnricher` mirrors it in the runner)
before applying them frozen to test. F5 applies the identical story in the latent space:

- **distance thresholds** (`latent-nn-dist`, `latent-centroid-dist`) — an anomaly score from
  distance to the legitimate reference (nearest neighbors / centroid), the continuous analogue of
  "the probability exceeds its cutoff";
- **angle thresholds** (`latent-cosine-centroid`) — the per-modality blocks are L2-normalized, so
  the informative geometry is angular; the score is `1 - cos` to the legitimate mean direction;
- **anomaly within clusters** (`latent-cluster-anom`) — k-means fitted on the LEGITIMATE fitting
  rows only (the FOC-179 leakage discipline), test rows scored by distance to the nearest center;
- **distributional thresholds** (`latent-gmm-density`) — an explicit density model of the
  legitimate terrain (PCA then diagonal GMM, both fitted on legitimate fitting rows only), scored
  by negative log likelihood — the closest analogue of the dictionary model's distributional
  calibration over probabilities;
- **light classifier** (`latent-logistic`) — Mateusz's F5 decision (2026-09-01, closing the issue's
  "decide later"): one supervised logistic regression over the fused embedding, evaluated through
  the runner like any other arm.

Every scorer calibrates a train-percentile operating point inside `fit()`
(`arms_latent._LatentAnomalyBase.train_percentile_`); the runner's frozen threshold drives the
comparable metrics below, and the calibration study runs after the main tables.

In [ ]:
# --- latent-nn-dist / latent-centroid-dist / latent-cosine-centroid: all axes
PRIMARY_AXIS = 'random-grouped'
AXIS_ORDER = ('random-grouped', 'grouped', 'chronological')

rows_dist = [
    run_arm_on_axis(arm, axis, enriched, y, cv=False)
    for arm in F5_ARMS[:3]
    for axis in AXIS_ORDER
]
print_comparison_table(rows_dist, title='latent distance/angle scorers x axes (frozen threshold)')
for row in rows_dist:
    print(
        '%-22s %-14s test positives %-3d | PR-AUC %.4f [CI %.4f, %.4f] vs chance %.4f (delta %+.4f) -> %s'
        % (row['arm'], row['axis'], row['test_positives'], row['pr_auc'], row['pr_auc_ci_low'],
           row['pr_auc_ci_high'], row['chance_level'], row['pr_auc'] - row['chance_level'],
           'covers chance' if row['pr_auc_ci_low'] <= row['chance_level'] <= row['pr_auc_ci_high']
           else 'separates'))

In [ ]:
# --- latent-cluster-anom / latent-gmm-density / latent-logistic: all axes
rows_model = [
    run_arm_on_axis(arm, axis, enriched, y, cv=False)
    for arm in F5_ARMS[3:]
    for axis in AXIS_ORDER
]
print_comparison_table(rows_model, title='cluster/density/logistic latent arms x axes (frozen threshold)')
for row in rows_model:
    print(
        '%-22s %-14s test positives %-3d | PR-AUC %.4f [CI %.4f, %.4f] vs chance %.4f (delta %+.4f) -> %s'
        % (row['arm'], row['axis'], row['test_positives'], row['pr_auc'], row['pr_auc_ci_low'],
           row['pr_auc_ci_high'], row['chance_level'], row['pr_auc'] - row['chance_level'],
           'covers chance' if row['pr_auc_ci_low'] <= row['chance_level'] <= row['pr_auc_ci_high']
           else 'separates'))

## Calibration study — the scorers' own train-percentile operating points

The runner freezes a best-F1 threshold on a validation carve so every arm is comparable; the
threshold scorers' own story — like the dictionary model's — is a threshold calibrated on the
training rows and applied frozen to test. Here each anomaly scorer is refit on the PRIMARY fit
carve exactly as the runner does, its legitimate-score percentiles (q95, q99) become the operating
points, and those FROZEN thresholds are applied to the held-out test rows. With 13 test positives
this is descriptive, not inferential — reported for completeness of the port.

In [ ]:
from sklearn.base import clone
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import train_test_split

PRIMARY = 'random-grouped'
train_idx, test_idx = axis_split(PRIMARY, enriched, y)
X_lat = _features_latent_pure(enriched)
X_tr, X_te = X_lat.loc[train_idx], X_lat.loc[test_idx]
y_tr, y_te = y.loc[train_idx], y.loc[test_idx]
X_fit, X_val, y_fit, y_val = train_test_split(
    X_tr, y_tr, test_size=0.25, random_state=42, stratify=y_tr
)
print('PRIMARY %s | fit carve %d rows (%d legit) | test %d rows, %d positives (chance %.4f)' % (
    PRIMARY, len(X_fit), int((y_fit == 0).sum()), len(X_te), int(y_te.sum()), float(y_te.mean())))

factories = [
    ('latent-nn-dist', arms_latent.NearestLegitScorer(n_neighbors=5, percentile=99.0)),
    ('latent-centroid-dist', arms_latent.CentroidScorer(percentile=99.0)),
    ('latent-cosine-centroid', arms_latent.CosineCentroidScorer(percentile=99.0)),
    ('latent-cluster-anom', arms_latent.ClusterAnomalyScorer(n_clusters=8, percentile=99.0)),
    ('latent-gmm-density', arms_latent.GMMDensityScorer(n_pca=64, n_components=4, percentile=99.0)),
]
calib_rows = []
for pct in (95.0, 99.0):
    for arm, est in factories:
        model = clone(est).set_params(percentile=pct)
        model.fit(X_fit, y_fit)
        scores_te = model.predict_proba(X_te)[:, 1]
        flags = scores_te >= model.train_percentile_
        calib_rows.append({
            'arm': arm, 'percentile': pct,
            'threshold': float(model.train_percentile_),
            'flagged': int(flags.sum()),
            'precision': float(precision_score(y_te, flags, zero_division=0)),
            'recall': float(recall_score(y_te, flags, zero_division=0)),
        })
calib = pd.DataFrame(calib_rows)
print('test positives: %d | chance level: %.4f | 13-arm context: all cover chance on PRIMARY' % (int(y_te.sum()), float(y_te.mean())))
print(calib.to_string(index=False, float_format=lambda v: '%.4f' % v))

## Comparison against the conceptual ancestor + the 13-arm context

The `dictionary` arm is the conceptual ancestor of this phase: its threshold logic is what F5 ports.
`xgb-baseline` and `latent-fusion` anchor the other ends (pure tabular model; the XGB arm over the
same latent space). Their rows come from the accumulated runner results (the F0-F4 CLI runs); the
F5 rows are this notebook's measurements on identical splits.

In [ ]:
prior = load_results(DEFAULT_RESULTS_PATH)
ANCHORS = ['dictionary', 'xgb-baseline', 'latent-fusion']
rows_f5 = rows_dist + rows_model
for axis in AXIS_ORDER:
    anchor_rows = [r for r in prior if r.get('axis') == axis and r.get('arm') in ANCHORS]
    f5_rows = [r for r in rows_f5 if r['axis'] == axis]
    combined = sorted(f5_rows + anchor_rows, key=lambda r: (r['arm'] not in ANCHORS, r['arm']))
    print_comparison_table(
        combined,
        title='%s — F5 threshold/statistical arms vs anchors (frozen threshold)' % axis,
    )
    print()

In [ ]:
# --- determinism: identity-aligned feature check + re-run equality ----------
# F3 round-2 lesson: NEVER compare feature frames positionally. Both builds
# below run on the same enriched frame; assert index equality FIRST, then diff.
lat_a = _features_latent_pure(enriched)
lat_b = _features_latent_pure(enriched)
assert lat_a.index.equals(lat_b.index), 'frame indices differ - cannot align'
delta = float(np.abs(lat_a.to_numpy(dtype=np.float64) - lat_b.to_numpy(dtype=np.float64)).max())
print('identity-aligned latent frame rebuild: max |delta| = %.1e over %d rows x %d dims' % (delta, lat_a.shape[0], lat_a.shape[1]))
assert delta == 0.0

r1 = run_arm_on_axis('latent-cluster-anom', PRIMARY_AXIS, enriched, y, cv=False)
r2 = run_arm_on_axis('latent-cluster-anom', PRIMARY_AXIS, enriched, y, cv=False)
del r1['axis'], r2['axis']
assert r1 == r2, 're-run of the same arm on the same split diverged'
print('latent-cluster-anom re-run: %d metric fields byte-identical (fixed seeds, no wall-clock fields)' % len(r1))

## Honest interpretation

- **Pre-registered null holds.** On the PRIMARY axis the fused latent space carries no detectable
  signal (F4: every embedding arm covers chance; fusion PR-AUC 0.0174 vs chance 0.0133). The
  threshold layer reads that same representation — distances, angles, clusters and densities over
  it — so it cannot manufacture signal the representation does not contain. Any F5 arm separating
  from chance here should be treated as an artifact first (see the chronological-axis caveat from
  F4: identity memorization, not signal).
- **What the machinery buys.** The dictionary model's calibration story (train-only thresholds,
  frozen at test) now exists for the latent space, inside the one runner protocol — new data can be
  dropped in and every arm re-compared without a parallel evaluation path.
- **Synthetic-data caveat.** Faces, text and demographics are simulated (F4). Conclusions are about
  method viability, not real-world fraud rates.
- **Scope.** F5 only; follow-ups are listed in the FOC-179 hand-off, not built here.